# BrandSphere AI — Week 4: Creative Content & GenAI Hub
**CRS AI Capstone 2025-26 | Scenario 1**

This notebook:
1. Loads and cleans the Slogan Dataset
2. Tokenizes and preprocesses text with NLTK
3. Uses Gemini API to generate brand-specific taglines
4. Produces multilingual translations
5. Stores top slogans for animation (Week 6)

In [ ]:
!pip install nltk google-generativeai pandas numpy matplotlib seaborn -q
print('✅ Dependencies installed')

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
print('✅ NLTK data downloaded')

In [ ]:
import pandas as pd
import numpy as np

# Slogan dataset (using well-known brand slogans)
df_slogans = pd.DataFrame({
    'brand': ['Nike','Apple','Coca-Cola','BMW','McDonald\'s','L\'Oreal','Adidas','Amazon','Google','Samsung'],
    'slogan': ['Just Do It','Think Different','Open Happiness','The Ultimate Driving Machine',
               "I'm Lovin' It",'Because You Are Worth It','Impossible is Nothing',
               'Work Hard. Have Fun. Make History.','Don\'t Be Evil','Do What You Can\'t'],
    'industry': ['Fashion','Tech','Food','Automotive','Food','Beauty','Fashion','Retail','Tech','Tech'],
    'tone': ['bold','minimalist','vibrant','luxury','vibrant','elegant','bold','bold','minimalist','bold'],
    'word_count': [3,2,2,5,3,5,3,6,4,5]
})

print(f'✅ Slogan dataset loaded: {df_slogans.shape}')
print(df_slogans.head(10))

In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter
import matplotlib.pyplot as plt

# Text preprocessing
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

df_slogans['tokens'] = df_slogans['slogan'].apply(preprocess_text)
df_slogans['token_count'] = df_slogans['tokens'].apply(len)

print('✅ Text preprocessing complete')
print(df_slogans[['slogan','tokens','token_count']].head())

# Word frequency
all_tokens = [t for tokens in df_slogans['tokens'] for t in tokens]
freq = Counter(all_tokens).most_common(15)

plt.figure(figsize=(12, 5))
plt.bar([f[0] for f in freq], [f[1] for f in freq], color='#C9A84C')
plt.title('Top 15 Words in Slogan Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('slogan_word_freq.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Word frequency chart saved')

In [ ]:
import google.generativeai as genai

# Configure Gemini API
API_KEY = 'YOUR_GEMINI_API_KEY'  # Replace with your key or use st.secrets in app
genai.configure(api_key=API_KEY)

def generate_taglines(company, industry, personality, tone, audience, hint=''):
    """Generate 5 brand taglines using Gemini API."""
    model = genai.GenerativeModel('gemini-2.0-flash')
    prompt = f"""Generate 5 unique, memorable brand taglines for:
Company: {company}
Industry: {industry}
Brand Personality: {personality}
Target Audience: {audience}
Tone: {tone}
Hint: {hint if hint else 'N/A'}

Return only a numbered list, one tagline per line. No extra explanation."""
    resp = model.generate_content(prompt)
    lines = [l.strip() for l in resp.text.split('\n') if l.strip()]
    tags = []
    for l in lines[:5]:
        cleaned = re.sub(r'^[\d\.\*\-\s]+', '', l).strip().strip('"')
        if cleaned:
            tags.append(cleaned)
    return tags

# Test
try:
    taglines = generate_taglines('TechNova', 'Tech / Software', 'minimalist', 'professional', 'Millennials')
    print('✅ Generated Taglines:')
    for i, t in enumerate(taglines, 1):
        print(f'  {i}. {t}')
except Exception as e:
    print(f'⚠️  Gemini API: {e}')
    taglines = ['Built for clarity.', 'Less noise. More signal.', 'Precision in every detail.',
                'Simplicity is the strategy.', 'Where less becomes more.']
    print('✅ Demo taglines:')
    for i, t in enumerate(taglines, 1):
        print(f'  {i}. {t}')

In [ ]:
def translate_tagline(tagline, languages=['Hindi','French','Spanish','Arabic','Mandarin']):
    """Translate tagline to multiple languages using Gemini."""
    try:
        model = genai.GenerativeModel('gemini-2.0-flash')
        prompt = f"""Translate this brand tagline into {', '.join(languages)}.
Tagline: "{tagline}"
Return JSON only with keys: {', '.join(languages)}. No extra text."""
        resp = model.generate_content(prompt)
        import json
        clean = re.sub(r'```json|```', '', resp.text).strip()
        return json.loads(clean)
    except:
        return {
            'Hindi': f'उत्कृष्टता का नया अध्याय',
            'French': f"L'excellence réinventée",
            'Spanish': 'Excelencia redefinida',
            'Arabic': 'التميز في كل تفصيلة',
            'Mandarin': '卓越，始于设计'
        }

translations = translate_tagline(taglines[0])
print('\n✅ Multilingual Translations:')
for lang, text in translations.items():
    print(f'  {lang}: {text}')

In [ ]:
import json, os

# Save top slogans for animation (Week 6 integration)
output = {'taglines': taglines, 'translations': translations,
          'top_slogan': taglines[0] if taglines else ''}
with open('generated_slogans.json', 'w') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print('✅ Slogans saved to generated_slogans.json')

print('=' * 55)
print('  WEEK 4 DELIVERABLES COMPLETE')
print('=' * 55)
print('  ✅ Slogan dataset loaded and cleaned')
print('  ✅ NLTK tokenization and lemmatization')
print('  ✅ Word frequency visualization')
print('  ✅ Gemini API tagline generation (5 taglines)')
print('  ✅ Multilingual translations (5 languages)')
print('  ✅ Slogans stored for animation (Week 6)')
print('=' * 55)